# 10 — Modelos baseados em grafos (TEA): GCN, GAT e GraphSAGE (opcional)

**Ordem de prioridade:** GCN primeiro (obrigatório); GAT se houver tempo (`RODAR_GAT`); GraphSAGE é um extra opcional (`RODAR_SAGE = False` por padrão). Nenhum deles é prometido como melhor que os baselines: com sinal fraco nos dados, o resultado honesto pode ficar perto do acaso.

**Decisões de construção do grafo (todas explícitas)**

| Decisão | Escolha | Observação |
|---|---|---|
| Unidade | **Um grafo por participante** | Sem grafo populacional com treino e teste juntos (evita vazamento por estrutura) |
| Nós | As 200 ROIs (inferidas do tamanho de `X`) | |
| Atributos do nó | Linha da matriz em **Fisher z, com sinal** (tamanho = nº de ROIs) | Padronizados com média/desvio **só do treino** do fold externo |
| Arestas | As `k%` maiores |r| do participante (10% ou 20%) | Densidade escolhida **só pela validação interna** |
| Correlações negativas | Entram pela magnitude |r|; o **sinal fica nos atributos do nó** | Alternativa (arestas assinadas) não testada |
| Diagonal | Removida; autoconexão adicionada só no passo de mensagem (GCN/GAT) | |
| Peso da aresta | |r| na propagação do GCN; atenção aprendida no GAT; média simples no GraphSAGE | |
| Classificação | Leitura do grafo (média + máximo dos nós) → MLP → 1 saída por participante | Em lotes de grafos densos (B × n × n) |
| Explicação | A atenção do GAT **não** é explicação biológica definitiva | Só hipótese exploratória (notebook 11) |

**Protocolo**
- Mesmos 5 folds externos por site do notebook 07; normalização só no treino; validação interna agrupada por site (a mesma checagem de viabilidade dos notebooks 08/09) para early stopping, **escolha da densidade (10% vs 20%)** e do limiar.
- Hiperparâmetros fixos e pequenos (sem grade grande): 2 camadas, 32 unidades, dropout 0,5, AdamW (lr 1e-3, wd 1e-3). Várias sementes. A tabela longa guarda tudo.
- Os folds são validação cruzada por site, não um teste independente.

In [ ]:
# 1. Projeto, módulo de avaliação e dataset (TEA / ABIDE I)
import sys, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

_raiz = Path.cwd().resolve()
for _c in (_raiz, *_raiz.parents):
    if (_c / 'src' / 'config.py').exists():
        _raiz = _c
        break
else:
    raise RuntimeError('Não encontrei src/config.py. Abra a pasta TCC_TEA_TDAH no VS Code.')
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src import avaliacao as av
criar_pastas()

NOME_BASE = 'TEA'
ROTULO_CASO = 'TEA'
ARQ_DATASET = PATHS['processed'] / 'dataset_abide_processado.pkl'
PASTA_RES = Path(PATHS['results'])
PASTA_MOD = Path(PATHS['models'])
PASTA_RES.mkdir(parents=True, exist_ok=True)
PASTA_MOD.mkdir(parents=True, exist_ok=True)

# 2. Parâmetros desta execução
import copy
import torch, torch.nn as nn, torch.nn.functional as F

DENSIDADES = [0.10, 0.20]
SEMENTES = [42, 43, 44]
RODAR_GAT = True
RODAR_SAGE = False            # extra opcional
HID, DROP, LR, WD = 32, 0.5, 1e-3, 1e-3
EPOCAS_MAX, PACIENCIA, BATCH, EPOCAS_FIXAS = 80, 12, 32, 30
if av.RAPIDO:
    DENSIDADES, SEMENTES, EPOCAS_MAX, PACIENCIA, EPOCAS_FIXAS = [0.10, 0.20], [42], 3, 2, 3
    print('*** TCC_RAPIDO=1: teste curto ***')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
d = av.carregar_dataset(ARQ_DATASET)
N = av.n_rois_de(d['X'].shape[1])
print(f'Base: {NOME_BASE} | participantes: {len(d["y"])} | ROIs (nós): {N} | dispositivo: {DEVICE}')
print('Classes:', pd.Series(d['y']).value_counts().sort_index().to_dict(), '| sites:', len(np.unique(d['sites'])))

_i, _j = np.triu_indices(N, k=1)
Z = np.zeros((len(d['X']), N, N), dtype=np.float32)       # Fisher z (com sinal), diagonal = 0
Z[:, _i, _j] = d['X']; Z[:, _j, _i] = d['X']
ABSR = np.abs(np.tanh(Z)).astype(np.float32)               # |r| (a diagonal já é 0)
EYE = np.eye(N, dtype=bool)
print('Z:', Z.shape, '| memória aproximada: %.0f MB' % (2 * Z.nbytes / 1e6))

def construir_arestas(dens):
    """Top-k% das |r| de cada participante (triângulo superior), simétrico, sem diagonal."""
    lim = np.quantile(np.abs(d['X']), 1 - dens, axis=1)
    return ((np.abs(Z) >= lim[:, None, None]) & ~EYE).astype(np.uint8)   # |z| e |r| têm a mesma ordem

ARESTAS = {dens: construir_arestas(dens) for dens in DENSIDADES}
for dens, A in ARESTAS.items():
    print(f'densidade {dens:.0%}: arestas por grafo = {int(A[0].sum() // 2)} (esperado ~{int(round(dens * N * (N - 1) / 2))}); simétrico: {bool((A == A.transpose(0, 2, 1)).all())}')

In [ ]:
# 3. Camadas e modelos densos (um lote = B grafos de N nós)
class CamadaGCN(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.lin = nn.Linear(i, o)
    def forward(self, H, A, W):
        # A_hat = D^-1/2 (A*|r| + I) D^-1/2
        At = W * A + torch.eye(H.shape[1], device=H.device)
        dinv = At.sum(-1).clamp(min=1e-6).pow(-0.5)
        return torch.bmm(dinv.unsqueeze(-1) * At * dinv.unsqueeze(-2), self.lin(H))

class CamadaGAT(nn.Module):
    def __init__(self, i, o, heads=4):
        super().__init__()
        self.h, self.o = heads, o
        self.W = nn.Linear(i, heads * o, bias=False)
        self.a_s = nn.Parameter(torch.randn(heads, o) * 0.1); self.a_d = nn.Parameter(torch.randn(heads, o) * 0.1)
        self.drop = nn.Dropout(0.3)
    def forward(self, H, A, W):
        B, n, _ = H.shape
        Hw = self.W(H).view(B, n, self.h, self.o)                                   # (B,n,h,o)
        es = (Hw * self.a_s).sum(-1); ed = (Hw * self.a_d).sum(-1)                  # (B,n,h)
        e = F.leaky_relu(es.permute(0, 2, 1).unsqueeze(-1) + ed.permute(0, 2, 1).unsqueeze(-2), 0.2)   # (B,h,n,n)
        mascara = (A + torch.eye(n, device=A.device)) > 0
        e = e.masked_fill(~mascara.unsqueeze(1), float('-inf'))
        att = self.drop(torch.softmax(e, dim=-1))
        out = torch.matmul(att, Hw.permute(0, 2, 1, 3))                             # (B,h,n,o)
        return out.permute(0, 2, 1, 3).reshape(B, n, self.h * self.o)

class CamadaSAGE(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.self_ = nn.Linear(i, o); self.viz = nn.Linear(i, o, bias=False)
    def forward(self, H, A, W):
        grau = A.sum(-1, keepdim=True).clamp(min=1.0)
        return self.self_(H) + self.viz(torch.bmm(A / grau, H))                     # média dos vizinhos

class RedeGrafo(nn.Module):
    def __init__(self, tipo, n_in, hid=HID, drop=DROP):
        super().__init__()
        self.tipo = tipo
        if tipo == 'GCN':
            self.l1, self.l2, d_out = CamadaGCN(n_in, hid), CamadaGCN(hid, hid), hid
        elif tipo == 'GAT':
            self.l1, self.l2 = CamadaGAT(n_in, hid // 4, 4), CamadaGAT(hid, hid // 4, 4); d_out = hid
        elif tipo == 'GraphSAGE':
            self.l1, self.l2, d_out = CamadaSAGE(n_in, hid), CamadaSAGE(hid, hid), hid
        else:
            raise ValueError(tipo)
        self.drop = nn.Dropout(drop)
        self.mlp = nn.Sequential(nn.Linear(2 * d_out, 32), nn.ReLU(), nn.Dropout(drop), nn.Linear(32, 1))
    def forward(self, Xf, A, W):
        H = self.drop(F.elu(self.l1(Xf, A, W)) if self.tipo == 'GAT' else F.relu(self.l1(Xf, A, W)))
        H = F.elu(self.l2(H, A, W)) if self.tipo == 'GAT' else F.relu(self.l2(H, A, W))
        leitura = torch.cat([H.mean(1), H.max(1).values], dim=1)                    # nível do grafo (participante)
        return self.mlp(self.drop(leitura)).squeeze(1)

def lote(FEATS, dens, idx):
    return (torch.as_tensor(FEATS[idx]).to(DEVICE),
            torch.as_tensor(ARESTAS[dens][idx].astype(np.float32)).to(DEVICE),
            torch.as_tensor(ABSR[idx]).to(DEVICE))

@torch.no_grad()
def prever_g(modelo, FEATS, dens, idx, batch=BATCH):
    modelo.eval(); out = []
    for k in range(0, len(idx), batch):
        out.append(torch.sigmoid(modelo(*lote(FEATS, dens, idx[k:k + batch]))).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def treinar_g(modelo, FEATS, dens, y, idx_tr, idx_va, semente):
    g = torch.Generator().manual_seed(semente)
    modelo = modelo.to(DEVICE)
    n_pos, n_neg = int((y[idx_tr] == 1).sum()), int((y[idx_tr] == 0).sum())
    perda_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([n_neg / max(n_pos, 1)], device=DEVICE))
    opt = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=WD)
    tem_val = idx_va is not None and len(np.unique(y[idx_va])) == 2
    epocas = EPOCAS_MAX if tem_val else EPOCAS_FIXAS
    melhor, estado, sem_melhora, rodadas = -1.0, None, 0, 0
    for ep in range(epocas):
        modelo.train(); rodadas += 1
        ordem = idx_tr[torch.randperm(len(idx_tr), generator=g).numpy()]
        for k in range(0, len(ordem), BATCH):
            b = ordem[k:k + BATCH]
            if len(b) < 2: continue
            opt.zero_grad(set_to_none=True)
            perda = perda_fn(modelo(*lote(FEATS, dens, b)), torch.as_tensor(y[b], dtype=torch.float32, device=DEVICE))
            perda.backward(); opt.step()
        if tem_val:
            auc = av.roc_auc_score(y[idx_va], prever_g(modelo, FEATS, dens, idx_va))
            if auc > melhor + 1e-4: melhor, estado, sem_melhora = auc, copy.deepcopy(modelo.state_dict()), 0
            else: sem_melhora += 1
            if sem_melhora >= PACIENCIA: break
    if tem_val and estado is not None: modelo.load_state_dict(estado)
    return modelo, (melhor if tem_val else np.nan), rodadas

def limiar_interno(y_va, p_va):
    if y_va is None or len(np.unique(y_va)) < 2: return 0.5
    cand = np.unique(np.round(p_va, 4))
    return float(max((av.balanced_accuracy_score(y_va, (p_va >= t).astype(int)), -abs(t - 0.5), t) for t in cand)[2])

def normalizar_atributos(idx_tr):
    """Média/desvio por coluna do atributo, só com participantes e nós do treino."""
    v = Z[idx_tr].reshape(-1, N)
    mu, sd = v.mean(0), v.std(0) + 1e-6
    return ((Z - mu) / sd).astype(np.float32)

In [ ]:
# 4. Validação cruzada externa agrupada por site
print('FOLDS EXTERNOS'); display(av.resumo_folds(d))
tipos = ['GCN'] + (['GAT'] if RODAR_GAT else []) + (['GraphSAGE'] if RODAR_SAGE else [])
folds = sorted(np.unique(d['fold_id']).tolist())
proba_oof = {m: np.full(len(d['y']), np.nan, dtype=np.float32) for m in tipos}
pred_oof = {m: np.full(len(d['y']), -1, dtype=np.int8) for m in tipos}
linhas, escolhas = [], []
(PASTA_MOD / 'grafos_folds').mkdir(exist_ok=True)

for fold in folds:
    te = np.where(d['fold_id'] == fold)[0]; tr = np.where(d['fold_id'] != fold)[0]
    splits, info = av.splits_internos(d['y'][tr], d['sites'][tr], k_max=3, seed=SEED)
    in_tr, in_va = (tr[splits[0][0]], tr[splits[0][1]]) if info['viavel'] else (tr, None)
    print(f'\n=== Fold {fold}: treino={len(tr)} (interno {len(in_tr)}, validação interna {0 if in_va is None else len(in_va)}) teste={len(te)}')
    FEATS = normalizar_atributos(tr)                  # só treino externo
    for tipo in tipos:
        for sem in SEMENTES:
            t0 = av.agora(); cand = {}
            for dens in (DENSIDADES if in_va is not None else [DENSIDADES[0]]):   # sem validação interna: 1ª densidade
                torch.manual_seed(sem * 100 + fold)
                m, auc_val, ep = treinar_g(RedeGrafo(tipo, N), FEATS, dens, d['y'], in_tr, in_va, sem)
                cand[dens] = (auc_val, m, ep)
            if in_va is not None:
                dens_esc = max(cand, key=lambda k: (np.nan_to_num(cand[k][0], nan=-1), -k))   # empate -> menor densidade
            else:
                dens_esc = DENSIDADES[0]
            auc_val, m, ep = cand[dens_esc]
            p_va = None if in_va is None else prever_g(m, FEATS, dens_esc, in_va)
            lim = limiar_interno(None if in_va is None else d['y'][in_va], p_va)
            p = prever_g(m, FEATS, dens_esc, te); pr = (p >= lim).astype(np.int8)
            lin = av.linha_resultado(NOME_BASE, tipo, fold, sem, d['y'][te], pr, p, limiar=lim, densidade=dens_esc,
                                     auc_val_interna=auc_val, epocas=ep, k_interno=int(info['k_usado']),
                                     seg_total=round(av.agora() - t0, 1))
            linhas.append(lin)
            escolhas.append(dict(fold=fold, modelo=tipo, semente=sem, **{f'auc_val_{k:.0%}': v[0] for k, v in cand.items()}, escolhida=dens_esc))
            if sem == SEMENTES[0]:
                proba_oof[tipo][te], pred_oof[tipo][te] = p, pr
                torch.save(dict(estado=m.state_dict(), densidade=dens_esc, tipo=tipo), PASTA_MOD / 'grafos_folds' / f'{tipo.lower()}_fold{fold}.pt')
            alerta = ' <-- ALERTA: uma única classe prevista' if lin['classe_unica_prevista'] else ''
            print(f"  {tipo:9s} sem={sem} dens={dens_esc:.0%} AUC={lin['auc_roc']:.3f} BalAcc={lin['balanced_accuracy']:.3f} "
                  f"AUCval={auc_val:.3f} limiar={lim:.2f} ep={ep} ({lin['seg_total']:.0f}s){alerta}")
    pd.DataFrame(linhas).to_csv(PASTA_RES / '10_resultados_long.csv', index=False)   # checkpoint

for m in tipos:
    if np.any(pred_oof[m] < 0) or not np.isfinite(proba_oof[m]).all():
        raise RuntimeError(f'Previsões OOF incompletas para {m}.')
df_long = pd.DataFrame(linhas); df_long['tipo'] = 'externo'

In [ ]:
# 5. Resumo e salvamento
res = av.resumir_resultados(df_long, d, proba_oof)
pd.set_option('display.width', 200)
print('MÉDIA ± DP ENTRE FOLDS'); display(res['por_fold'])
print('\nMÉDIA PONDERADA PELO TAMANHO DO FOLD'); display(res['ponderado'])
print('\nAUC OOF (1ª semente) com IC 95% por bootstrap de sites'); display(res['auc_oof_ic'])
print('\nDENSIDADE ESCOLHIDA NA VALIDAÇÃO INTERNA (contagem por modelo)')
display(df_long.groupby(['modelo', 'densidade']).size().unstack(fill_value=0))
print('Colapso (uma classe prevista) em', int(df_long['classe_unica_prevista'].sum()), 'de', len(df_long), 'execuções.')
print('Critério a priori: IC inferior > 0,5 => "acima do acaso" (não é prova). A atenção do GAT não é explicação biológica.')

df_long.to_csv(PASTA_RES / '10_resultados_long.csv', index=False)
pd.DataFrame(escolhas).to_csv(PASTA_RES / '10_escolha_densidade.csv', index=False)
res['por_fold'].to_csv(PASTA_RES / '10_resumo_por_fold.csv')
res['ponderado'].to_csv(PASTA_RES / '10_resumo_ponderado.csv', index=False)
res['auc_oof_ic'].to_csv(PASTA_RES / '10_auc_oof_ic.csv', index=False)
av.salvar_oof(PASTA_RES / '10_predicoes_oof.npz', d, proba_oof, pred_oof)
print('Salvo em:', PASTA_RES)